# ElectroPatch: eight-complex comparison in Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smaiti7/electropatch/blob/main/colab/ElectroPatch_Comparison.ipynb)

Run the **same Python package and frozen PDB inputs** used by the local command-line workflow. This notebook clones your public GitHub repository, executes `electropatch compare` and `electropatch ml`, and displays both comparisons and individual pose plots. The score is a heuristic; basic-phosphate contacts are **not** an electrostatic energy calculation.

**How to use:** Open this notebook in Colab and choose **Runtime → Run all**. The public repository URL is already set in the first code cell. No GPU is needed. Set `STORAGE_MODE` in the first code cell to choose Google Drive, temporary runtime storage, or automatic cleanup. For the scientific method and limitations, read the repository README.

## Choose where results go

Set `STORAGE_MODE` in the next cell:

- `"temporary"` (default): keep CSV, JSON and figures in this Colab session; the final cell also creates a downloadable ZIP. They disappear when the runtime is deleted.
- `"drive"`: mount your Google Drive and save each run under `MyDrive/ElectroPatch/runs/`. Colab will request Drive access. The files persist after the runtime ends.
- `"off"`: calculate and display the results, then delete generated CSV, JSON and figures in the final cell. Nothing is saved to Drive or zipped. Re-run all cells to explore files again.

All modes need a working directory while calculations run; `"off"` removes it at the end.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from uuid import uuid4

# Change the repository URL only if you are running a fork.
REPO_URL = "https://github.com/smaiti7/electropatch"
STORAGE_MODE = "temporary"  # Choose "temporary", "drive", or "off".
DRIVE_FOLDER = "ElectroPatch"  # Used only when STORAGE_MODE is "drive".
if STORAGE_MODE not in {"temporary", "drive", "off"}:
    raise ValueError("STORAGE_MODE must be temporary, drive, or off")

candidates = [Path.cwd(), Path.cwd().parent, Path("/content/electropatch")]
REPO_ROOT = next((p.resolve() for p in candidates if (p / "data" / "manifest.csv").is_file()), None)
if REPO_ROOT is None:
    url = REPO_URL.strip() or input("Paste the public GitHub URL for electropatch: ").strip()
    if not url.startswith("https://github.com/"):
        raise ValueError("Use the public https://github.com/USER/electropatch URL")
    clone_base = Path("/content") if Path("/content").exists() else Path(tempfile.mkdtemp())
    REPO_ROOT = clone_base / "electropatch"
    subprocess.run(["git", "clone", "--depth", "1", url, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT) + "[ml]", "pandas"], check=True)
if STORAGE_MODE == "drive":
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Google Drive mode requires a Google Colab runtime") from exc
    drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "runs"
    drive_root.mkdir(parents=True, exist_ok=True)
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
    WORK_DIR = drive_root / run_id
    WORK_DIR.mkdir()
else:
    runtime_base = "/content" if Path("/content").is_dir() else None
    WORK_DIR = Path(tempfile.mkdtemp(prefix="electropatch-output-", dir=runtime_base))
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True, stderr=subprocess.DEVNULL).strip()
except subprocess.CalledProcessError:
    commit = "uncommitted local checkout"
print("Repository:", REPO_ROOT)
print("Commit:", commit)
print("Storage mode:", STORAGE_MODE)
print("Output:", WORK_DIR)


## Run the frozen comparison

This executes all eight manifest-selected complexes with 50 decoys each, seed 42, and checksum verification. The run writes separate results and figures for each structure.

In [ ]:
subprocess.run([
    sys.executable, "-m", "electropatch", "compare",
    "--manifest", str(REPO_ROOT / "data" / "manifest.csv"),
    "--output-dir", str(WORK_DIR),
    "--n-decoys", "50", "--seed", "42",
], check=True)


## Compare native ranks

A smaller rank is better. **Do not compare raw combined scores between structures:** each score uses z-standardization within its own 51-pose pool.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.read_csv(WORK_DIR / "results" / "comparison.csv")
comparison[["pdb_id", "system_name", "native_rank_geometry", "native_rank_combined", "rank_change_with_contact_descriptor"]]


In [ ]:
image = plt.imread(WORK_DIR / "figures" / "comparison.png")
fig, ax = plt.subplots(figsize=(13, 8))
ax.imshow(image)
ax.axis("off")
plt.show()


## Inspect all eight pose-ranking curves

Each panel shows scores within one structure. The orange bar is the experimental native pose. The panels are a visual check of within-system rankings, not an across-system score comparison.

In [ ]:
ids = comparison["pdb_id"].tolist()
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=False)
for ax, pdb_id in zip(axes.flat, ids):
    frame = pd.read_csv(WORK_DIR / "results" / pdb_id / "pose_scores.csv").sort_values("rank")
    colors = ["#df6b2d" if value == "native" else "#3d6482" for value in frame["pose_id"]]
    ax.bar(frame["rank"], frame["combined_score"], color=colors, width=0.9)
    native_rank = int(frame.loc[frame["pose_id"] == "native", "rank"].iloc[0])
    ax.set(title=f"{pdb_id}: native {native_rank}/{len(frame)}", xlabel="Pose rank")
    ax.axhline(0, color="0.5", linewidth=0.8)
    ax.grid(axis="y", alpha=0.2)
axes[0, 0].set_ylabel("Within-system combined score (z units)")
fig.tight_layout()
plt.show()


## Explore one complex

Change `PDB_ID` to any ID in the comparison table, then rerun this cell. It shows the package-generated ranking and feature plots plus the top-scoring poses.

In [ ]:
PDB_ID = "1AAY"  # Choose any PDB ID from the table above.
if PDB_ID not in ids:
    raise ValueError(f"Choose one of {ids}")
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, filename in zip(axes, ("pose_ranking.png", "feature_comparison.png")):
    ax.imshow(plt.imread(WORK_DIR / "figures" / PDB_ID / filename))
    ax.axis("off")
fig.tight_layout()
plt.show()
pose_table = pd.read_csv(WORK_DIR / "results" / PDB_ID / "pose_scores.csv")
pose_table.sort_values("rank")["pose_id pose_type contacts clashes basic_phosphate_contacts mean_nearest_distance geometry_rank rank".split()].head(10)


## Leave-one-complex-out ML ranking

This optional model uses the same four descriptors. A StandardScaler and logistic classifier are fitted on seven complexes and rank all poses in the eighth; this repeats once per held-out complex. The result is a synthetic-decoy pilot, not external docking validation. See `docs/ANALYSIS.md` for interpretation.

In [ ]:
subprocess.run([
    sys.executable, "-m", "electropatch", "ml",
    "--manifest", str(REPO_ROOT / "data" / "manifest.csv"),
    "--output-dir", str(WORK_DIR),
], check=True)
ml_table = pd.read_csv(WORK_DIR / "results" / "ml_comparison.csv")
display(ml_table[["pdb_id", "system_name", "native_rank_combined", "native_rank_ml", "training_complexes"]])
fig, ax = plt.subplots(figsize=(13, 8))
ax.imshow(plt.imread(WORK_DIR / "figures" / "ml_comparison.png"))
ax.axis("off")
plt.show()


## Finish: save or discard generated files

Run this cell after viewing the tables and figures. In `temporary` mode it creates a ZIP in the Colab **Files** panel. In `drive` mode your output directory is already in Google Drive. In `off` mode it removes the runtime files; displayed notebook outputs remain visible.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

if STORAGE_MODE == "off":
    shutil.rmtree(WORK_DIR)
    print("Generated files deleted. Re-run all cells to regenerate them.")
elif STORAGE_MODE == "drive":
    print("Results saved in Google Drive:", WORK_DIR)
else:
    archive = WORK_DIR.parent / f"{WORK_DIR.name}.zip"
    with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
        for directory in ("results", "figures"):
            for path in sorted((WORK_DIR / directory).rglob("*")):
                if path.is_file():
                    bundle.write(path, path.relative_to(WORK_DIR))
    print("Download from the Colab Files panel:", archive)


The eight systems are an exploratory pilot, not an accuracy benchmark. Decoys are rigid, can clash, and omit solvent and conformational relaxation. A later PDB2PQR/APBS study would require structure preparation, parameter choices, and sensitivity checks.